# 🔬 Laboratorio de Distribuciones Continuas Avanzadas y de Inferencia
**Asignatura:** Probabilidad y Estadística

Este notebook complementario explora distribuciones continuas fundamentales en confiabilidad, modelado de fenómenos físicos e inferencia estadística:
1. **Exponencial** (Tiempo de vida sin memoria)
2. **t de Student** (Inferencia con muestras pequeñas)
3. **Chi-cuadrada ($chi^2$)** (Pruebas de bondad de ajuste y varianza)
4. **Laplace / Doble Exponencial** (Modelado de colas pesadas y errores)
5. **Rayleigh** (Magnitudes de vectores 2D, viento y señales)
6. **Weibull** (Análisis de falla y confiabilidad de sistemas)

---

In [ ]:
# Configuración e importación de dependencias
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from ipywidgets import interact, fixed

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 5.5)
plt.rcParams['font.size'] = 10

print('✅ Entorno listo para trabajar con distribuciones avanzadas.')

--- 
## 1. Distribución Exponencial: $X \sim \text{Exp}(\lambda)$

### 💡 Ejemplos de uso e aplicación real:
- **Sistemas de colas y servicios:** Tiempo entre llegadas de clientes a una ventanilla o servidores web.
- **Confiabilidad:** Tiempo transcurrido hasta la primera falla en componentes electrónicos (bajo la suposición de tasa de falla constante / falta de memoria).
- **Física:** Tiempo transcurrido entre desintegraciones radiactivas consecutivas de un isótopo.

- **PDF:** $f(x) = \lambda e^{-\lambda x}, \quad x \ge 0$
- **Esperanza:** $E[X] = \frac{1}{\lambda}$, **Varianza:** $Var(X) = \frac{1}{\lambda^2}$

In [ ]:
def lab_exponencial(lam=1.0, N=1000, seed=42):
    np.random.seed(seed)
    scale = 1.0 / lam
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    x = np.linspace(0, stats.expon.ppf(0.999, scale=scale), 500)
    
    pdf = stats.expon.pdf(x, scale=scale)
    cdf = stats.expon.cdf(x, scale=scale)
    sample = stats.expon.rvs(scale=scale, size=N)
    
    # Muestrales vs Teóricos
    mean_sample, var_sample = np.mean(sample), np.var(sample)
    mean_teor, var_teor = scale, scale**2
    
    # PDF e Histograma
    axs[0].hist(sample, bins=35, density=True, alpha=0.4, color='#e67e22', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#2c3e50', linewidth=2.5, label='PDF Teórica')
    axs[0].set_title(f'Exponencial(λ={lam}) — E[X]={mean_teor:.2f} | x̄={mean_sample:.2f}')
    axs[0].set_xlabel('Tiempo / Intervalo (x)')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # CDF
    axs[1].plot(x, cdf, color='#27ae60', linewidth=2.5, label='CDF Teórica F(x)')
    axs[1].set_title(f'CDF — Var Teórica={var_teor:.2f} | s²={var_sample:.2f}')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x) = P(X <= x)')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_exponencial,
         lam=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='Tasa (λ):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

--- 
## 2. Distribución t de Student: $X \sim t(\nu)$

### 💡 Ejemplos de uso e aplicación real:
- **Inferencia Estadística:** Construcción de intervalos de confianza y pruebas de hipótesis para la media muestral cuando la varianza poblacional $\sigma^2$ es **desconocida** y la muestra es pequeña ($n < 30$).
- **Econometría y Finanzas:** Modelado de retornos de activos financieros que presentan "colas pesadas" (mayor probabilidad de eventos extremos que una Normal).

- **Parámetro clave:** Grados de libertad $\nu = n - 1$.
- **Propiedad:** Cuando $\nu \to \infty$, $t(\nu) \to \mathcal{N}(0, 1)$.

In [ ]:
def lab_tstudent(df=3, N=1000, seed=42):
    np.random.seed(seed)
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    x = np.linspace(-5, 5, 500)
    
    pdf_t = stats.t.pdf(x, df=df)
    pdf_norm = stats.norm.pdf(x, loc=0, scale=1) # Referencia Normal Standar
    cdf_t = stats.t.cdf(x, df=df)
    sample = stats.t.rvs(df=df, size=N)
    
    # PDF e Histograma
    axs[0].hist(sample, bins=35, density=True, alpha=0.4, color='#3498db', edgecolor='black', label=f'Muestra t (N={N})')
    axs[0].plot(x, pdf_t, color='#2c3e50', linewidth=2.5, label=f't-Student (ν={df})')
    axs[0].plot(x, pdf_norm, 'r--', linewidth=1.5, label='Normal(0,1) de referencia')
    axs[0].set_title(f't-Student (ν={df}) vs Normal Standard')
    axs[0].set_xlabel('t')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # CDF
    axs[1].plot(x, cdf_t, color='#27ae60', linewidth=2.5, label='CDF t-Student')
    axs[1].plot(x, stats.norm.cdf(x), 'r--', linewidth=1.5, label='CDF Normal(0,1)')
    axs[1].set_title('CDF — Comparación de Colas')
    axs[1].set_xlabel('t')
    axs[1].set_ylabel('F(t)')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_tstudent,
         df=widgets.IntSlider(min=1, max=50, step=1, value=3, description='G.L. (ν):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

--- 
## 3. Distribución Chi-cuadrada: $X \sim \chi^2(k)$

### 💡 Ejemplos de uso e aplicación real:
- **Inferencia sobre Varianzas:** Estimación e intervalos de confianza para la varianza $\sigma^2$ de una población normal.
- **Pruebas de Bondad de Ajuste:** Evaluación de si una muestra observada proviene de una distribución teórica específica.
- **Pruebas de Independencia:** Tablas de contingencia en estudios sociológicos y epidemiológicos.

- **Definición:** Es la suma de los cuadrados de $k$ variables aleatorias normales independientes normalizadas: $X = \sum_{i=1}^k Z_i^2$.
- **Esperanza:** $E[X] = k$, **Varianza:** $Var(X) = 2k$.

In [ ]:
def lab_chi2(k=3, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(0, stats.chi2.ppf(0.999, df=k), 500)
    pdf = stats.chi2.pdf(x, df=k)
    cdf = stats.chi2.cdf(x, df=k)
    sample = stats.chi2.rvs(df=k, size=N)
    
    # PDF e Histograma
    axs[0].hist(sample, bins=35, density=True, alpha=0.4, color='#9b59b6', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#2c3e50', linewidth=2.5, label=f'Chi²(k={k})')
    axs[0].set_title(f'Chi-cuadrada(k={k}) — Esperanza Teórica={k} | x̄={np.mean(sample):.2f}')
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # CDF
    axs[1].plot(x, cdf, color='#27ae60', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title(f'CDF — Var Teórica={2*k} | s²={np.var(sample):.2f}')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x)')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_chi2,
         k=widgets.IntSlider(min=1, max=30, step=1, value=3, description='G.L. (k):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

--- 
## 4. Distribución de Laplace (Doble Exponencial): $X \sim \text{Laplace}(\mu, b)$

### 💡 Ejemplos de uso e aplicación real:
- **Procesamiento de Señales e Imágenes:** Modelado de coeficientes de transformada de Fourier/Wavelet.
- **Machine Learning:** Fundamento teórico de la regularización $L_1$ (Lasso regression).
- **Finanzas:** Modelado de cambios en precios de acciones cuando hay saltos bruscos o picos pronunciados alrededor de cero.

- **PDF:** $f(x) = \frac{1}{2b} \exp\left(-\frac{|x-\mu|}{b}\right)$
- **Características:** Curva pico simétrica no diferenciable en su punto máximo ($x=\mu$).

In [ ]:
def lab_laplace(mu=0.0, b=1.0, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(mu - 6*b, mu + 6*b, 500)
    pdf = stats.laplace.pdf(x, loc=mu, scale=b)
    cdf = stats.laplace.cdf(x, loc=mu, scale=b)
    sample = stats.laplace.rvs(loc=mu, scale=b, size=N)
    
    # PDF e Histograma
    axs[0].hist(sample, bins=35, density=True, alpha=0.4, color='#1abc9c', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#c0392b', linewidth=2.5, label=f'Laplace(μ={mu}, b={b})')
    axs[0].set_title(f'Laplace(μ={mu}, b={b}) — Densidad (PDF)')
    axs[0].set_xlabel('x')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # CDF
    axs[1].plot(x, cdf, color='#27ae60', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title('Acumulada (CDF)')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x)')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_laplace,
         mu=widgets.FloatSlider(min=-5.0, max=5.0, step=0.5, value=0.0, description='Centro (μ):'),
         b=widgets.FloatSlider(min=0.2, max=4.0, step=0.2, value=1.0, description='Escala (b):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

--- 
## 5. Distribución Rayleigh: $X \sim \text{Rayleigh}(\sigma)$

### 💡 Ejemplos de uso e aplicación real:
- **Energía Eólica:** Modelado de la velocidad horizontal del viento.
- **Telecomunicaciones:** Amplitud de señales que sufren desvanecimiento (Rayleigh fading) al propagarse por múltiples trayectorias.
- **Física Vectorial:** Magnitud $R = \sqrt{X^2 + Y^2}$ de un vector 2D donde $X$ e $Y$ son variables normales independientes $N(0, \sigma^2)$.

In [ ]:
def lab_rayleigh(scale=1.0, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(0, stats.rayleigh.ppf(0.999, scale=scale), 500)
    pdf = stats.rayleigh.pdf(x, scale=scale)
    cdf = stats.rayleigh.cdf(x, scale=scale)
    sample = stats.rayleigh.rvs(scale=scale, size=N)
    
    mean_teor = scale * np.sqrt(np.pi / 2)
    
    # PDF e Histograma
    axs[0].hist(sample, bins=35, density=True, alpha=0.4, color='#34495e', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#e74c3c', linewidth=2.5, label=f'Rayleigh(σ={scale})')
    axs[0].set_title(f'Rayleigh(σ={scale}) — E[X]={mean_teor:.2f} | x̄={np.mean(sample):.2f}')
    axs[0].set_xlabel('Magnitud (x)')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # CDF
    axs[1].plot(x, cdf, color='#27ae60', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title('Acumulada (CDF)')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x)')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_rayleigh,
         scale=widgets.FloatSlider(min=0.2, max=5.0, step=0.2, value=1.0, description='Escala (σ):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));

--- 
## 6. Distribución de Weibull: $X \sim \text{Weibull}(\beta, \eta)$

### 💡 Ejemplos de uso e aplicación real:
- **Ingeniería de Confiabilidad:** Generaliza la función de tasa de falla:
  - $\beta < 1$: Tasa de falla decreciente ("mortalidad infantil" de componentes).
  - $\beta = 1$: Tasa de falla constante (coincide con la distribución **Exponencial**).
  - $\beta > 1$: Tasa de falla creciente (desgaste y envejecimiento de materiales).
- **Meteorología:** Modelado preciso de ráfagas extremas de viento en parques eólicos.

- **Parámetros:** Forma $\beta$ (`c` en SciPy) y escala $\eta$ (`scale` en SciPy).

In [ ]:
def lab_weibull(beta=1.5, eta=2.0, N=1000, seed=42):
    np.random.seed(seed)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(0.001, stats.weibull_min.ppf(0.999, c=beta, scale=eta), 500)
    pdf = stats.weibull_min.pdf(x, c=beta, scale=eta)
    cdf = stats.weibull_min.cdf(x, c=beta, scale=eta)
    sample = stats.weibull_min.rvs(c=beta, scale=eta, size=N)
    
    # PDF e Histograma
    axs[0].hist(sample, bins=35, density=True, alpha=0.4, color='#d35400', edgecolor='black', label=f'Muestra (N={N})')
    axs[0].plot(x, pdf, color='#2c3e50', linewidth=2.5, label=f'Weibull(β={beta}, η={eta})')
    axs[0].set_title(f'Weibull(β={beta}, η={eta}) — Densidad (PDF)')
    axs[0].set_xlabel('Tiempo de vida (x)')
    axs[0].set_ylabel('Densidad')
    axs[0].legend()
    
    # CDF
    axs[1].plot(x, cdf, color='#27ae60', linewidth=2.5, label='CDF Teórica')
    axs[1].set_title('Acumulada (CDF)')
    axs[1].set_xlabel('x')
    axs[1].set_ylabel('F(x)')
    axs[1].legend()
    
    plt.tight_layout()
    plt.show()

interact(lab_weibull,
         beta=widgets.FloatSlider(min=0.4, max=5.0, step=0.2, value=1.5, description='Forma (β):'),
         eta=widgets.FloatSlider(min=0.5, max=10.0, step=0.5, value=2.0, description='Escala (η):'),
         N=widgets.SelectionSlider(options=[30, 100, 500, 1000, 5000, 20000], value=1000, description='Muestra (N):'),
         seed=fixed(42));